> ## SOLUTION / ANSWER KEY &mdash; Lab 10.2
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-02-least-privilege.ipynb`](../lab-02-least-privilege.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 10.2 &mdash; Least Privilege

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 5 &middot; Module 10 &mdash; Ethics &amp; Responsible AI in Agentic Systems**

### What you'll do
- Grant only the tools the task actually needs
- Never grant a dangerous (consequential) tool
- Check a tool grant respects least privilege

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it &amp; observe**. The responsible-AI logic here &mdash; injection defence, least privilege, trace-reading, fairness, the eval loop, the guardrails &mdash; is **real, deterministic Python** you can run offline. The **Advanced agent labs (10&ndash;12)** additionally drive a **real Groq model** through `create_agent`: **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a responsible agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The agent labs use a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`); the guardrail/eval/trace logic is genuine rule-based Python. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. This is the **course finale** &mdash; Lab 5.2: responsible-AI frameworks &amp; **debugging agent errors**.

**Reference:** [Module 10 slides &mdash; Safety: guardrails, consolidated](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 10 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = "/tmp/biaa-lab-10-02"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=5):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
The strongest, simplest safety control (deck slides 8&ndash;9): **least privilege**. Give the agent only
the tools its task needs, and **never** a dangerous one &mdash; it cannot misuse a capability it doesn't
have. This limits the **blast radius** when something goes wrong, and it neutralises injection: a hijacked
agent can't do what it **can't**. (You'll enforce this on a *real* Groq agent in Labs 11&ndash;12 &mdash; the
tools list simply never includes a trade/send tool.)

In [ ]:
DANGEROUS = {"place_trade", "delete_records", "send_email", "wire_funds", "isolate_host"}
print("consequential tools to withhold unless truly required:", DANGEROUS)

## Build it
Complete `grant_tools` (only needed **and** safe) and `is_least_privilege`. Notice the task *claims*
it needs `send_email` &mdash; least privilege withholds it anyway.

In [ ]:
DANGEROUS = {"place_trade", "delete_records", "send_email", "wire_funds", "isolate_host"}

def grant_tools(needed, catalog):
    # least privilege: grant only tools that are needed AND not dangerous
    return [t for t in catalog if t in needed and t not in DANGEROUS]

def is_least_privilege(granted, needed):
    # granted must be a subset of needed AND contain no dangerous tool
    return set(granted) <= set(needed) and not any(t in DANGEROUS for t in granted)

try:
    catalog = ["lookup", "compute", "send_email", "summarize"]
    needed  = ["lookup", "compute", "send_email"]   # note: task claims it "needs" send_email
    granted = grant_tools(needed, catalog)
    print("granted        :", granted)
    print("least privilege?:", is_least_privilege(granted, needed))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- A **dangerous** tool is withheld **even when the task says it needs it** &mdash; the policy overrides the request.
- `grant_tools` returns a **subset** of what was asked, never a superset &mdash; the agent can't reach beyond its job.
- This is the same guardrail you'll apply for real in Labs 11&ndash;12: the create_agent tools list is read-only, so no `place_trade`.

## Your turn (open task &mdash; no grader)
Add a new dangerous capability (e.g. `"refund"`) to `DANGEROUS` and a task that "needs" it. **What good
looks like:** `grant_tools` still refuses it, and `is_least_privilege` flags any grant that sneaks a dangerous
tool in. Ask yourself: for your own capstone agent, which single tool would you never bind?

---
### Key takeaway
Grant only what the task needs, never the dangerous tool. The capability an agent doesn't have cannot be misused -- by a bug, a bad reasoning step, or a hijack. Least privilege is the recurring strongest guardrail of the whole course.

[&#8592; All Module 10 labs](./index.html) &nbsp;&middot;&nbsp; [Module 10 slides](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>